# 1B · E-wallet fraud — final pipeline
Feature engineering + small residual MLP (PyTorch), 5-fold x 3-seed ensemble, F1-optimal threshold chosen on out-of-fold predictions.
CV (OOF) F1 ≈ 0.570 · public 0.5751 · private 0.5884 (Gold).

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import StratifiedKFold

torch.set_num_threads(1)

DATA = Path("/home/minh/Desktop/olympiad_ai/warmup/1B_fraud_classification/dataset")
OUT = Path("/home/minh/Desktop/olympiad_ai/solutions/1B")
TARGET = "is_fraud"
CFG = dict(hidden=64, depth=1, drop=0.2, lr=2e-3, wd=1e-4, epochs=8, bs=512, folds=5, seeds=3)

CATS = ["e_commerce", "electronics", "food_delivery", "game_topup",
        "gift_card", "p2p_transfer", "travel", "utilities"]
CHANNELS = ["app", "qr", "web"]

## 1. Data
Key EDA findings (train, 59,854 rows, 4.66% fraud, no missing values, random id split):
- `amount / avg_amount_30d` is **U-shaped**: log-ratio < -3 → 97% fraud (card testing), > 2 → 31%.
- Amount < 20k VND → 27% fraud; amounts that are multiples of 50k → ~10-11% (2.5x base rate).
- Account age ≤ 30 days → ~15% fraud (sharp step), ages > 10 years are impossible outliers → capped.
- `tx_count_24h ≥ 5`, `failed_pin_24h`, `new_device`, night hours (0-5) all raise risk; distance only matters > 100 km.
- Category x channel interaction: web channel roughly doubles risk within each category (gift_card/web 29%).

In [2]:
train = pd.read_csv(DATA / 'train/train.csv')
public = pd.read_csv(DATA / 'public_test/public_test.csv')
private = pd.read_csv(DATA / 'private_test/private_test.csv')
print(train.shape, public.shape, private.shape, 'fraud rate', train[TARGET].mean().round(4))
lr = np.log(train.amount_vnd / train.avg_amount_30d)
print(train.groupby(pd.cut(lr, [-9,-3,-2,-1,0,1,2,9]), observed=True)[TARGET].agg(['mean','size']))
print(train.groupby(train.amount_vnd % 50000 == 0)[TARGET].agg(['mean','size']))
print(train.groupby(train.account_age_days <= 30)[TARGET].agg(['mean','size']))
print(train.pivot_table(index='merchant_category', columns='channel', values=TARGET).round(3))

(59854, 13) (10058, 12) (10088, 12) fraud rate 0.0466
              mean   size
(-9, -3]  0.966102    236
(-3, -2]  0.307692    481
(-2, -1]  0.022631   5921
(-1, 0]   0.018146  23531
(0, 1]    0.040383  23525
(1, 2]    0.136748   5799
(2, 9]    0.307479    361
                mean   size
amount_vnd                 
False       0.040233  54010
True        0.105749   5844
                      mean   size
account_age_days                 
False             0.040518  56543
True              0.151012   3311
channel              app     qr    web
merchant_category                     
e_commerce         0.027  0.029  0.064
electronics        0.074  0.063  0.149
food_delivery      0.010  0.010  0.017
game_topup         0.107  0.101  0.215
gift_card          0.149  0.154  0.292
p2p_transfer       0.055  0.071  0.141
travel             0.029  0.031  0.090
utilities          0.004  0.003  0.007


In [3]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-wise feature engineering (no statistics learned from data -> no leakage)."""
    a, avg = df.amount_vnd, df.avg_amount_30d
    f = pd.DataFrame(index=df.index)
    f["log_amt"] = np.log(a)
    f["log_avg"] = np.log(avg)
    f["log_ratio"] = np.log(a / avg)                    # amount vs the user's own 30d average
    f["ratio_pos"] = f.log_ratio.clip(lower=0)          # U-shaped risk: unusually large ...
    f["ratio_neg"] = (-f.log_ratio).clip(lower=0)       # ... or unusually small (card testing)
    f["tiny_amt"] = (a < 20000).astype(float)           # micro-payments: 27% fraud vs 4.7%
    f["round_50k"] = (a % 50000 == 0).astype(float)     # round amounts: ~2.5x fraud rate
    f["round_10k"] = (a % 10000 == 0).astype(float)
    f["hour_sin"] = np.sin(2 * np.pi * df.hour / 24)
    f["hour_cos"] = np.cos(2 * np.pi * df.hour / 24)
    f["night"] = (df.hour <= 5).astype(float)
    f["is_weekend"] = df.is_weekend.astype(float)
    f["log_age"] = np.log1p(df.account_age_days.clip(upper=3650))  # cap impossible ages (>10y)
    f["young"] = (df.account_age_days <= 30).astype(float)          # step: 15% vs 4% fraud
    f["tx"] = df.tx_count_24h.clip(upper=12).astype(float)
    f["tx_hi"] = (df.tx_count_24h >= 5).astype(float)
    f["log_dist"] = np.log1p(df.distance_from_home_km.clip(upper=1000))
    f["far"] = (df.distance_from_home_km > 100).astype(float)
    f["new_device"] = df.new_device.astype(float)
    f["failed_pin"] = df.failed_pin_24h.clip(upper=4).astype(float)
    f["cat"] = df.merchant_category.map({c: i for i, c in enumerate(CATS)}).fillna(0).astype(int)
    f["ch"] = df.channel.map({c: i for i, c in enumerate(CHANNELS)}).fillna(0).astype(int)
    return f

X, Xpu, Xpr = make_features(train), make_features(public), make_features(private)
y = train[TARGET].values
X.head()

,log_amt,log_avg,log_ratio,ratio_pos,ratio_neg,tiny_amt,round_50k,round_10k,hour_sin,hour_cos,...,log_age,young,tx,tx_hi,log_dist,far,new_device,failed_pin,cat,ch
0,12.276997,12.541115,-0.264118,0.000000,0.264118,0.0,0.0,0.0,0.707107,-7.071068e-01,...,4.934474,0.0,0.0,0.0,1.308333,0.0,0.0,0.0,0,0
1,13.582948,12.432411,1.150537,1.150537,0.000000,0.0,0.0,0.0,-0.707107,7.071068e-01,...,5.123964,0.0,3.0,0.0,1.791759,0.0,0.0,0.0,0,1
2,13.053228,12.615530,0.437698,0.437698,0.000000,0.0,0.0,0.0,-0.707107,-7.071068e-01,...,8.202756,0.0,3.0,0.0,0.182322,0.0,0.0,0.0,0,2
3,13.562294,14.197571,-0.635276,0.000000,0.635276,0.0,0.0,0.0,-1.000000,-1.836970e-16,...,6.767343,0.0,3.0,0.0,1.987874,0.0,0.0,0.0,0,0
4,13.802020,13.617669,0.184351,0.184351,0.000000,0.0,0.0,0.0,-0.866025,5.000000e-01,...,5.129899,0.0,4.0,0.0,3.931826,0.0,0.0,0.0,5,1


## 2. Model
Numeric features (standardised per fold) + embeddings for merchant category (4-d), channel (2-d) and their cross (4-d) → Linear(64) → 1 pre-activation residual block (BatchNorm, SiLU, dropout 0.2) → BN/SiLU/Linear head. Plain BCE: class weights / focal loss did not improve F1 because the threshold is tuned anyway.

In [4]:
class ResBlock(nn.Module):
    def __init__(self, h: int, drop: float) -> None:
        super().__init__()
        self.bn, self.l1, self.l2, self.drop = nn.BatchNorm1d(h), nn.Linear(h, h), nn.Linear(h, h), nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.l2(self.drop(F.silu(self.l1(F.silu(self.bn(x))))))


class FraudNet(nn.Module):
    """Numeric features + embeddings for category, channel and category x channel."""

    def __init__(self, n_num: int, hidden: int, depth: int, drop: float) -> None:
        super().__init__()
        self.e_cat, self.e_ch, self.e_x = nn.Embedding(8, 4), nn.Embedding(3, 2), nn.Embedding(24, 4)
        self.inp = nn.Linear(n_num + 10, hidden)
        self.blocks = nn.Sequential(*[ResBlock(hidden, drop) for _ in range(depth)])
        self.head = nn.Sequential(nn.BatchNorm1d(hidden), nn.SiLU(), nn.Linear(hidden, 1))

    def forward(self, xn: torch.Tensor, cat: torch.Tensor, ch: torch.Tensor) -> torch.Tensor:
        z = torch.cat([xn, self.e_cat(cat), self.e_ch(ch), self.e_x(cat * 3 + ch)], dim=1)
        return self.head(self.blocks(self.inp(z))).squeeze(-1)

print(FraudNet(len(X.columns) - 2, CFG['hidden'], CFG['depth'], CFG['drop']))

FraudNet(
  (e_cat): Embedding(8, 4)
  (e_ch): Embedding(3, 2)
  (e_x): Embedding(24, 4)
  (inp): Linear(in_features=30, out_features=64, bias=True)
  (blocks): Sequential(
    (0): ResBlock(
      (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (l1): Linear(in_features=64, out_features=64, bias=True)
      (l2): Linear(in_features=64, out_features=64, bias=True)
      (drop): Dropout(p=0.2, inplace=False)
    )
  )
  (head): Sequential(
    (0): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): SiLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 3. Training & Inference
AdamW + OneCycle LR (max 2e-3), only 8 epochs (longer training overfits: 30 epochs dropped OOF F1 to 0.53). 5-fold stratified CV x 3 seeds = 15 models. The threshold is chosen on the out-of-fold predictions (smoothed F1 curve), and test predictions average all 15 fold models so their calibration matches the OOF used to pick the threshold.

In [5]:
def fit_predict(Xa: pd.DataFrame, ya: np.ndarray, X_eval: list, seed: int) -> list:
    torch.manual_seed(seed)
    np.random.seed(seed)
    num = [c for c in Xa.columns if c not in ("cat", "ch")]
    mu, sd = Xa[num].mean(), Xa[num].std() + 1e-6       # scaler fitted on the training fold only

    def to_t(d: pd.DataFrame):
        return (torch.tensor(((d[num] - mu) / sd).values, dtype=torch.float32),
                torch.tensor(d.cat.values), torch.tensor(d.ch.values))

    xa, ca, ha = to_t(Xa)
    yt = torch.tensor(ya, dtype=torch.float32)
    model = FraudNet(len(num), CFG["hidden"], CFG["depth"], CFG["drop"])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    n, bs = len(ya), CFG["bs"]
    steps = (n + bs - 1) // bs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=steps * CFG["epochs"], pct_start=0.2)
    for _ in range(CFG["epochs"]):
        model.train()
        perm = torch.randperm(n)
        for i in range(steps):
            idx = perm[i * bs:(i + 1) * bs]
            if len(idx) < 2:
                sched.step()
                continue
            opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(model(xa[idx], ca[idx], ha[idx]), yt[idx])
            loss.backward()
            opt.step()
            sched.step()
    model.eval()
    with torch.no_grad():
        return [torch.sigmoid(model(*to_t(d))).numpy() for d in X_eval]


def best_threshold(y: np.ndarray, p: np.ndarray) -> tuple:
    ts = np.quantile(p, np.linspace(0.85, 0.995, 300))
    f1s = np.array([f1_score(y, p >= t) for t in ts])
    f1s_smooth = np.convolve(f1s, np.ones(9) / 9, mode="same")  # robust to a lucky single cut
    i = int(np.argmax(f1s_smooth))
    return float(ts[i]), float(f1s[i])

In [6]:
oof = np.zeros(len(y)); p_pub = np.zeros(len(Xpu)); p_pri = np.zeros(len(Xpr))
n_models = CFG['folds'] * CFG['seeds']
skf = StratifiedKFold(CFG['folds'], shuffle=True, random_state=0)
for k, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    for s in range(CFG['seeds']):
        pv, pu, pr = fit_predict(X.iloc[tr_idx], y[tr_idx], [X.iloc[va_idx], Xpu, Xpr], seed=100 * k + s)
        oof[va_idx] += pv / CFG['seeds']; p_pub += pu / n_models; p_pri += pr / n_models
    print(f'fold {k}: AP={average_precision_score(y[va_idx], oof[va_idx]):.4f}')
thr, f1 = best_threshold(y, oof)
print(f'OOF AP={average_precision_score(y, oof):.4f}  OOF F1={f1:.4f} at threshold {thr:.3f}')

fold 0: AP=0.6087


fold 1: AP=0.6034


fold 2: AP=0.6098


fold 3: AP=0.5950


fold 4: AP=0.6596


OOF AP=0.6144  OOF F1=0.5699 at threshold 0.294


## 4. Predict

In [7]:
for name, ids, p in [('public', public.id, p_pub), ('private', private.id, p_pri)]:
    pred = (p >= thr).astype(int)
    pd.DataFrame({'id': ids, TARGET: pred}).to_csv(OUT / f'{name}_submission.csv', index=False)
    print(f'{name}: {len(pred)} rows, predicted fraud rate {pred.mean():.3%}')
pd.read_csv(OUT / 'public_submission.csv').head()

public: 10058 rows, predicted fraud rate 4.564%
private: 10088 rows, predicted fraud rate 4.629%


,id,is_fraud
0,5,0
1,7,0
2,24,0
3,30,0
4,37,0
